<a href="https://colab.research.google.com/github/sadiyatamanna/EdvergenceX-Learning/blob/main/Context_Engineering_Lab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Context Engineering – Hands-On Demo

## Objective
In this lab, we will learn how to:
- Structure context instead of dumping prompts
- Separate static, dynamic, and memory context
- Build a real-world, AI system

### Use Case
Customer Support AI for an EdTech platform

> Key idea: **Context is infrastructure, not text.**


## Problem Statement

We want to build an AI system that:
- Acts as a customer support agent
- Follows company refund policies strictly
- Responds politely and clearly
- Uses user-specific information to decide responses

This is **not a chatbot**.  
This is a **context-aware AI system**.

## The Wrong Way: Context Dumping

Many people put everything into one prompt.
This approach does not scale and is hard to control.

In [1]:
bad_prompt = """
You are a support agent.
Follow all rules.
Here is the refund policy...
User wants a refund.
"""
print(bad_prompt)


You are a support agent.
Follow all rules.
Here is the refund policy...
User wants a refund.



## Context Engineering Approach

We break context into structured parts:

1. Static Context → Who the AI is and how it behaves
2. External Context → Policies or documents
3. Dynamic Context → User input and task data
4. Memory Context → What the system remembers

We assemble these intentionally.

## Step 1: Static Context

Static context defines:
- The role of the AI
- Rules and constraints
- Tone and behavior

This rarely changes.

In [2]:
SYSTEM_CONTEXT = """
You are a customer support assistant for an EdTech platform.

Rules:
- Be polite and professional
- Do not promise refunds
- Follow company policies strictly
- Escalate to human support if unsure
"""

## Step 2: External Context (Policies)

The AI should not guess policies.
We explicitly provide them as context.

In [3]:
REFUND_POLICY = """
Refund Policy:
- Refunds are allowed only within 7 days of purchase
- Course progress must be below 20%
- Subscriptions are non-refundable
"""

## Step 3: Dynamic Context

Dynamic context changes per request.
This includes user input and user-specific data.

In [4]:
user_query = "I purchased the course 10 days ago and want a refund"

user_profile = {
    "role": "student",
    "course_progress": "15%",
    "purchase_days_ago": 10
}

## Step 4: Context Assembly

We now assemble the context carefully.
Order and clarity matter.
This is **context engineering**.

In [5]:
final_prompt = f"""
{SYSTEM_CONTEXT}

Company Policy:
{REFUND_POLICY}

User Profile:
- Role: {user_profile['role']}
- Course Progress: {user_profile['course_progress']}
- Purchased: {user_profile['purchase_days_ago']} days ago

User Question:
{user_query}
"""

In [6]:
print(final_prompt)



You are a customer support assistant for an EdTech platform.

Rules:
- Be polite and professional
- Do not promise refunds
- Follow company policies strictly
- Escalate to human support if unsure


Company Policy:

Refund Policy:
- Refunds are allowed only within 7 days of purchase
- Course progress must be below 20%
- Subscriptions are non-refundable


User Profile:
- Role: student
- Course Progress: 15%
- Purchased: 10 days ago

User Question:
I purchased the course 10 days ago and want a refund



In [8]:
from google.colab import userdata

# Access the API key stored in Colab Secrets
MY_API_KEY = userdata.get('api_key')

# Now, use this variable when initializing your OpenAI client
# client = OpenAI(api_key=MY_API_KEY, base_url="https://apidev.navigatelabsai.com")

print("API key loaded successfully from Colab Secrets.")
print("Please update the client initialization in the cell above (hok-nnOpgi2r) to use `api_key=MY_API_KEY`.")

API key loaded successfully from Colab Secrets.
Please update the client initialization in the cell above (hok-nnOpgi2r) to use `api_key=MY_API_KEY`.


## Step 5: Call the Model

We now send the structured context to the model.

In [11]:
from openai import OpenAI

client = OpenAI(api_key=MY_API_KEY, base_url="https://nexusapi.navigatelabs.ai")

response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {"role": "system", "content": SYSTEM_CONTEXT},
        {"role": "user", "content": final_prompt}
    ]
)

print(response.choices[0].message.content)

Thank you for reaching out. Based on our company's refund policy, refunds are available within 7 days of purchase and if the course progress is below 20%. Since your purchase was made 10 days ago and your course progress is at 15%, I am unable to process a refund at this time. 

If there's anything else I can assist you with, please let me know. For further assistance, I can escalate this matter to our human support team.


## Why Did the AI Respond Correctly?

The model:
- Refused the refund
- Followed policy
- Maintained a professional tone

This happened **because of context**, not because the model is smart.

## Step 6: Memory Context

Real AI systems remember important user information.
We store **summarized memory**, not full chat history.

In [12]:
SESSION_MEMORY = """
User previously asked about course difficulty.
User is price-sensitive.
"""

## Context Assembly with Memory

We now include session memory into the context.

In [13]:
final_prompt_with_memory = f"""
{SYSTEM_CONTEXT}

Session Memory:
{SESSION_MEMORY}

Company Policy:
{REFUND_POLICY}

User Question:
{user_query}
"""

In [14]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": SYSTEM_CONTEXT},
        {"role": "user", "content": final_prompt_with_memory}
    ]
)

print(response.choices[0].message.content)

Thank you for reaching out. I understand you're requesting a refund for a course purchased 10 days ago.

According to our company policy, refunds are allowed only within 7 days of the purchase date, and course progress must be below 20%. As your purchase was made 10 days ago, it falls outside of our 7-day refund window.

Therefore, we are unable to process a refund in this instance.

Is there anything else I can assist you with regarding your course or platform experience?


## Your Task

1. Choose a real-world domain
2. Define:
   - Static context
   - Dynamic context
   - Memory context
3. Assemble context in code
4. Show one real workflow
5. Explain how this scales to your capstone

No generic chatbots allowed.

# 1. Real-World Domain

Domain: College Course Assistance

# 2. Define the Four Contexts
**Static Context**

Information that rarely changes.

In [15]:
STATIC_CONTEXT = """
You are a College Course Assistance AI.

Rules:
- Give answers related to the student's enrolled courses.
- Explain concepts clearly at an undergraduate level.
- Do not invent course policies or deadlines.
- If required information is unavailable, clearly say so.
- Use the student's academic context when answering.
"""

**External Context**

Course information supplied by the college.

In [16]:
COURSE_CONTEXT = """
Course: Artificial Intelligence

Topics:
1. Introduction to AI
2. Machine Learning
3. Neural Networks
4. Convolutional Neural Networks
5. Natural Language Processing

Assessment:
- Assignment: 20%
- Mid-Semester Exam: 30%
- End-Semester Exam: 50%

Current Unit:
Unit 4 - Convolutional Neural Networks
"""

**Dynamic Context**

Information that changes for every request.

In [17]:
student_profile = {
    "name": "Student",
    "semester": 7,
    "department": "Computer Science",
    "enrolled_course": "Artificial Intelligence",
    "current_unit": "CNN"
}

user_query = "Can you explain how pooling reduces the size of a feature map?"

**Memory Context**

Important information remembered from previous interactions.

In [18]:
MEMORY_CONTEXT = """
Previous learning information:
- Student prefers simple explanations.
- Student understands basic CNN concepts.
- Student previously learned convolution and stride.
- Student requested step-by-step numerical examples.
"""

# 3. Context Assembly in Code

This is the main context engineering part of the demo.

In [19]:
final_prompt = f"""
{STATIC_CONTEXT}

===== COURSE CONTEXT =====
{COURSE_CONTEXT}

===== STUDENT CONTEXT =====
Name: {student_profile['name']}
Semester: {student_profile['semester']}
Department: {student_profile['department']}
Enrolled Course: {student_profile['enrolled_course']}
Current Unit: {student_profile['current_unit']}

===== MEMORY CONTEXT =====
{MEMORY_CONTEXT}

===== CURRENT QUESTION =====
{user_query}

Answer the question using the provided academic context.
Explain it step-by-step at the student's level.
"""

print(final_prompt)



You are a College Course Assistance AI.

Rules:
- Give answers related to the student's enrolled courses.
- Explain concepts clearly at an undergraduate level.
- Do not invent course policies or deadlines.
- If required information is unavailable, clearly say so.
- Use the student's academic context when answering.


===== COURSE CONTEXT =====

Course: Artificial Intelligence

Topics:
1. Introduction to AI
2. Machine Learning
3. Neural Networks
4. Convolutional Neural Networks
5. Natural Language Processing

Assessment:
- Assignment: 20%
- Mid-Semester Exam: 30%
- End-Semester Exam: 50%

Current Unit:
Unit 4 - Convolutional Neural Networks


===== STUDENT CONTEXT =====
Name: Student
Semester: 7
Department: Computer Science
Enrolled Course: Artificial Intelligence
Current Unit: CNN

===== MEMORY CONTEXT =====

Previous learning information:
- Student prefers simple explanations.
- Student understands basic CNN concepts.
- Student previously learned convolution and stride.
- Student re

# 4. One Real Workflow

Use this workflow in your presentation:

**Student asks:**

"Can you explain how pooling reduces the size of a feature map?"

The system does not simply send the question to the AI.

# Example result

**Because the system knows:**

Student is studying Artificial Intelligence
Current topic is CNN
Student already understands convolution and stride
Student prefers simple, step-by-step explanations

the AI can answer at the appropriate level instead of giving a generic explanation.

**For example:**

Pooling reduces the spatial size of a feature map by selecting or combining values from small regions.

For example, with 2×2 max pooling and stride 2, a 6×6 feature map becomes a 3×3 feature map because the 2×2 window moves two positions at a time and selects the maximum value from each region.

# 5. Actual API Call

You can then connect the assembled context to the same API structure from your lab:

In [23]:
from openai import OpenAI

client = OpenAI(
    api_key=MY_API_KEY,
    base_url="Base_url"
)

response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {
            "role": "system",
            "content": STATIC_CONTEXT
        },
        {
            "role": "user",
            "content": final_prompt
        }
    ]
)

print(response.choices[0].message.content)

Certainly! Pooling is a technique used in Convolutional Neural Networks (CNNs) to reduce the size of the feature map while retaining important information. Here's a simple, step-by-step explanation:

### 1. **What is a feature map?**
- After applying a convolution, you get a feature map, which shows where certain features (like edges or textures) are detected in the image.
- This feature map is usually large, and processing it directly can be computationally expensive.

### 2. **Why reduce the size of a feature map?**
- To make the network faster and reduce overfitting.
- To focus on the most important features and ignore minor details.

### 3. **What does pooling do?**
- Pooling down-samples (reduces) the feature map by summarizing regions of it with a single value.
- The most common pooling methods are **max pooling** and **average pooling**.

### 4. **How does max pooling work?**
- Divide the feature map into small sections (called windows or slices), for example, a 2x2 area.
- For 

# 6. How This Scales to Your Capstone

The same context engineering approach can be expanded into the final College Course Assistance AI System.

Instead of keeping course information and student memory manually inside the prompt, the capstone can retrieve the required information from the student's academic data and previous interactions.

**Memory Context**

The system stores summarized student learning information, rather than the complete chat history.

In [27]:
STUDENT_MEMORY = """
Student previously studied Convolution and Stride.
Student prefers simple, step-by-step explanations.
Student has difficulty understanding Pooling.
Student previously requested numerical examples for CNN topics.
"""

**Context Assembly with Memory**

We now include the student's memory along with the course and current student context.

In [28]:
final_prompt_with_memory = f"""
{STATIC_CONTEXT}

Course Context:
{COURSE_CONTEXT}

Student Context:
- Semester: {student_profile['semester']}
- Department: {student_profile['department']}
- Current Course: {student_profile['enrolled_course']}
- Current Unit: {student_profile['current_unit']}

Student Memory:
{STUDENT_MEMORY}

Current Question:
{user_query}

Answer according to the student's course,
current learning level, and previous learning history.
"""

**AI Model Response**

The assembled context is then sent to the AI model.

In [29]:
response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {
            "role": "system",
            "content": STATIC_CONTEXT
        },
        {
            "role": "user",
            "content": final_prompt_with_memory
        }
    ]
)

print(response.choices[0].message.content)

Certainly! In the context of convolutional neural networks (CNNs), pooling is a operation used to reduce the size of a feature map, which helps make the model more efficient and reduces the chance of overfitting.

Here's a simple, step-by-step explanation, considering your background and previous studies:

1. **What is pooling?**  
Pooling is a process that takes a small region of the feature map and summarizes it into a single value. Common types are max pooling and average pooling.

2. **How does it work?**  
Imagine dividing the feature map into small, non-overlapping regions, like a grid. For example, a 2x2 region.

3. **Step-by-step example (using max pooling):**  
Let's say you have a 4x4 feature map:

\[ \begin{bmatrix}
1 & 3 & 2 & 4 \\
5 & 6 & 1 & 2 \\
1 & 2 & 4 & 0 \\
3 & 1 & 2 & 5 \\
\end{bmatrix} \]

- You apply 2x2 max pooling:
  - Divide the map into four 2x2 regions:
    - Top-left: \(\begin{bmatrix} 1 & 3 \\ 5 & 6 \end{bmatrix}\)
    - Top-right: \(\begin{bmatrix} 2 & 4 


This allows the capstone to move from a simple AI question-answering system to a context-aware academic assistance system that adapts its responses based on the student's course, current topic, and learning history.